# 28 — Soumission finale : blend rank champion × pseudo-labeling (25/75)

Pipeline complet de la soumission `28_rank_w75.csv` (équipe **Hackatsuki**).

## Recette
1. **Champion** : CatBoost (depth 6, lr 0.05, 600 itér., seed 42), restreint à `op_03`
   (100% de la fraude s'y trouve ; probabilité 0 ailleurs). Features : montants/ratios,
   incohérences de solde, fréquences de comptes, comportement par compte/paire, dynamique
   récente, et `te_origin` (taux de fraude historique de l'émetteur, encodé **fold-safe** en OOF).
2. **Pseudo-labeling 1 cycle** (cf. notebook 23) : les prédictions très confiantes du champion
   sur le test (proba > 0.98 → fraude, < 0.02 → légitime) sont ajoutées au train, puis
   réentraînement. Motivation : ~9.7% des comptes destinataires du test sont nouveaux ;
   le pseudo-labeling donne au modèle un signal sur ces comptes futurs.
3. **Blend par moyenne de rangs** : 25% champion + 75% modèle pseudo. L'AP est une métrique
   de rang → le rank-average surpasse la moyenne brute ; le poids 75% vient du balayage
   de pondérations (notebooks 24-26 et série 28).
4. **Aucune calibration** (elle dégrade l'AP d'environ 0.013, cf. notebooks 01-05).

Déterministe (seed 42 partout). Dépendances : `requirements.txt` à la racine ;
données attendues dans `data/` (`train.csv`, `test.csv`, `sample_submission.csv`).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from catboost import CatBoostClassifier
from src import config as C
from src.utils import op03_mask, seed_everything, make_submission
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv"); test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
op03 = op03_mask(train).to_numpy(); y_all = train[C.TARGET].to_numpy()
te_op = op03_mask(test).to_numpy()
EPS = 1e-6; WINDOWS = (5, 10, 20); SM = 30
print(f"train {len(train):,} | test {len(test):,} | op03 train {op03.sum():,} / test {te_op.sum():,}")

## Features (identiques au notebook 08, le modèle champion)

In [ ]:
def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]; f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)

def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        X[f"fq_{col}"] = df[col].map(ref[col].value_counts(normalize=True)).fillna(0).values
    return pd.concat([X,
                      behavioral_features(df, ref).reset_index(drop=True),
                      recency_features(df, ref).reset_index(drop=True),
                      recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)], axis=1)

def ftr(df, ref):
    X = base_build(df, ref)
    X["te"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM)
    return X

def fap(df, ref):
    X = base_build(df, ref)
    mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM)
    X["te"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm)
    return X

def make_cat():
    return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                              learning_rate=0.05, iterations=600, random_seed=42, verbose=False)

## Étape 1 — Champion : entraînement sur tout le train op_03, prédiction du test

In [ ]:
ref0 = train.iloc[np.where(op03)[0]]; y0 = y_all[op03]
test_op = test.iloc[np.where(te_op)[0]].copy()
pch = make_cat().fit(ftr(ref0, ref0), y0).predict_proba(fap(test_op, ref0))[:, 1]
print("champion entraîné | proba test op03 :", pch.shape)

## Étape 2 — Pseudo-labeling (1 cycle, seuils 0.98 / 0.02) et réentraînement

In [ ]:
mf = pch > 0.98; ml = pch < 0.02
pse = test_op.iloc[np.where(mf | ml)[0]].copy()
pse[C.TARGET] = (pch[mf | ml] > 0.98).astype(float)
aug = pd.concat([ref0, pse], ignore_index=True)
print(f"pseudo-labels : {int(mf.sum())} fraudes / {int(ml.sum())} légitimes | train augmenté {len(aug):,}")
p1 = make_cat().fit(ftr(aug, aug), aug[C.TARGET].to_numpy()).predict_proba(fap(test_op, aug))[:, 1]
print("modèle pseudo entraîné")

## Étape 3 — Rank-average 25% champion / 75% pseudo, et écriture de la soumission

In [ ]:
def rk(x):
    return np.argsort(np.argsort(x)) / (len(x) - 1)

WP = 0.75
blend = (1 - WP) * rk(pch) + WP * rk(p1)
full = np.zeros(len(test)); full[te_op] = blend
path = make_submission(test[C.ID], full, "28_rank_w75")
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print("soumission écrite :", path, "| proba>0 :", int((sub['target'] > 0).sum()))